# Endpoint Health Anomaly Detection

Detects CPU usage anomalies across a 5-server fleet using per-server statistical baselines, and investigates the two servers with patterns that warrant follow-up.

**What this does**
- Cleans invalid sensor readings and fills missing values
- Flags CPU anomalies per server using a mean plus 2 standard deviation threshold
- Runs detailed analysis on the two servers with the most significant findings
- Verifies isolated events against the full fleet before drawing conclusions

**Built with**
- pandas
- matplotlib

## Part 1: Load Data

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt

df = pd.read_csv("portfolio/Phase_2/endpoint_health_raw.csv")

## Part 2: Data Quality — Invalid Readings

`memory_gb` contains negative values, which are physically impossible. These are treated as missing data rather than dropped, so they can be recovered in the missing value handling step below.

In [ ]:
df.loc[df['memory_gb'] < 0, 'memory_gb'] = pd.NA

## Part 3: Missing Value Handling

In [ ]:
missing_before = df.isnull().sum()
missing_pct_before = (df.isnull().mean() * 100).round(2)

print("\nMissing before:")
print(missing_before)
print("\nMissing % before:")
print(missing_pct_before)

df.ffill(inplace=True)

missing_after = df.isnull().sum()
print("\nMissing after:")
print(f"{missing_after}\n")

## Part 4: srv-web-02 Anomaly Detection

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
web_server_df = df[df['server'] == 'srv-web-02'].copy()

cpu_mean = web_server_df['cpu_percent'].mean()
cpu_std = web_server_df['cpu_percent'].std()
threshold = cpu_mean + 2 * cpu_std

web_server_df['cpu_zscore'] = (web_server_df['cpu_percent'] - cpu_mean) / cpu_std

anomalies = web_server_df[web_server_df['cpu_percent'] > threshold].copy()
anomalies['severity'] = anomalies['cpu_zscore'].apply(
    lambda z: 'severe' if z >= 3 else 'borderline'
)

print(f"srv-web-02 CPU baseline: mean={cpu_mean:.1f}%, std={cpu_std:.1f}%")
print(f"Anomaly threshold (mean + 2 std): {threshold:.1f}%\n")
print(anomalies[['timestamp', 'cpu_percent', 'cpu_zscore', 'severity', 'memory_gb', 'network_mbps']])

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(web_server_df['timestamp'], web_server_df['cpu_percent'], color='red', label='CPU %')
plt.axhline(threshold, color='gray', linestyle='--', label=f'Threshold ({threshold:.1f}%)')
plt.scatter(anomalies['timestamp'], anomalies['cpu_percent'], color='black', zorder=5, label='Flagged')
plt.title('CPU Usage Over Time: srv-web-02')
plt.xlabel('Time')
plt.ylabel('CPU Percent')
plt.legend()
plt.grid(True)
plt.show()

## Part 5: Fleet-Wide Anomaly Detection

In [ ]:
df['cpu_zscore'] = df.groupby('server')['cpu_percent'].transform(
    lambda x: (x - x.mean()) / x.std()
)
df['cpu_threshold'] = df.groupby('server')['cpu_percent'].transform(
    lambda x: x.mean() + 2 * x.std()
)

fleet_anomalies = df[df['cpu_percent'] > df['cpu_threshold']].copy()
fleet_anomalies['severity'] = fleet_anomalies['cpu_zscore'].apply(
    lambda z: 'severe' if z >= 3 else 'borderline'
)

print(
    fleet_anomalies[['timestamp', 'server', 'cpu_percent', 'cpu_zscore', 'severity', 'memory_gb', 'network_mbps']]
    .sort_values(['server', 'timestamp'])
)

## Part 6: srv-app-02 Anomaly Detection

In [ ]:
app02_df = df[df['server'] == 'srv-app-02'].copy()

cpu_mean = app02_df['cpu_percent'].mean()
cpu_std = app02_df['cpu_percent'].std()
threshold = cpu_mean + 2 * cpu_std

app02_anomalies = app02_df[app02_df['cpu_percent'] > threshold].copy()
app02_anomalies['severity'] = app02_anomalies['cpu_zscore'].apply(
    lambda z: 'severe' if z >= 3 else 'borderline'
)

print(f"\nsrv-app-02 CPU baseline: mean={cpu_mean:.1f}%, std={cpu_std:.1f}%")
print(f"Anomaly threshold (mean + 2 std): {threshold:.1f}%\n")
print(app02_anomalies[['timestamp', 'cpu_percent', 'cpu_zscore', 'severity', 'memory_gb', 'network_mbps']])

### Revision 1: Bug Check - z-score column typo

The first version of the srv-app-02 block created its own local column, `cpu_zcore` (missing the s), instead of reusing the `cpu_zscore` column already computed by the fleet-wide `groupby().transform()` call in Part 5. Because `app02_df` inherited that fleet-wide column, `app02_anomalies['severity']` was already using the correct values by accident, but the print statement displayed the typo column instead, a different set of numbers than what actually drove the severity labels.

Fixed by removing the redundant local calculation entirely and using the fleet-wide `cpu_zscore` column for both the severity logic and the printed output, so there is only one z-score column and it matches what the reader sees.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(app02_df['timestamp'], app02_df['cpu_percent'], color='red', label='CPU %')
plt.axhline(threshold, color='gray', linestyle='--', label=f'Threshold ({threshold:.1f}%)')
plt.scatter(app02_anomalies['timestamp'], app02_anomalies['cpu_percent'], color='black', zorder=5, label='Flagged')
plt.title('CPU Usage Over Time: srv-app-02')
plt.xlabel('Time')
plt.ylabel('CPU Percent')
plt.legend()
plt.grid(True)
plt.show()

### Revision 2: Enhancement - removed leftover debug prints

The working script had two leftover print statements checking `app02_df['timestamp'].dtype` and `.head()`, left over from confirming `pd.to_datetime()` worked correctly. These were removed before finalizing the notebook, since dtype-checking output has no narrative purpose for a reader.

## srv-app-01 Verification

srv-app-01 shows one severe, isolated spike. It is captured in the fleet-wide table above rather than given its own detailed section. The check below confirms no other server was flagged at the same timestamp, which supports treating this as an isolated event.

In [ ]:
spike_time = fleet_anomalies[fleet_anomalies['server'] == 'srv-app-01']['timestamp'].iloc[0]

print(f"srv-app-01 spike timestamp: {spike_time}\n")
print(fleet_anomalies[fleet_anomalies['timestamp'] == spike_time])

## Findings

Across the 5-server fleet, CPU anomaly detection surfaced two servers with patterns worth flagging and two that read as statistical noise.

srv-app-02 is the most concerning result in this dataset. It logged 6 severe anomalies concentrated in a single 5-hour window on 06-01, from 2pm to 7pm. Memory and network stayed flat throughout, which rules out resource contention or a traffic spike as the driver and isolates the anomaly to CPU alone. The sustained plateau, rather than an isolated spike, is what separates this from normal variance and makes it worth a closer look at what was scheduled or running on that server during those hours.

srv-web-02 showed a different signature entirely. Its 2 severe spikes (05-21 and 05-26) plus 1 borderline point (06-06) were each isolated single-hour events, not a sustained pattern. What stands out is that CPU was not the leading indicator: network usage was elevated while memory stayed low, suggesting these events were network-driven rather than compute-driven, possibly a traffic burst or a data transfer rather than a processing load issue.

srv-app-01 had one severe isolated spike (171.08% on 06-02), also single-hour with no surrounding pattern. The verification above confirms no other server was flagged at that timestamp, so this reads as an isolated event rather than a fleet-wide issue.

srv-db-01 and srv-web-01 had borderline-only anomalies (9 and 17 respectively), scattered with no clustering in time. Given the volume of scattered points relative to severe events, these are treated as statistical noise rather than actionable findings, consistent with normal operational variance rather than a signal worth escalating.

Taken together, srv-app-02 is the priority for follow-up given the sustained, CPU-isolated nature of its anomaly window. srv-web-02's pattern is secondary but worth noting for its distinct network-driven signature. The remaining three servers do not show anomaly patterns that warrant action based on this analysis alone.